# Sistema di Raccomandazione Film con IBM LNN

Questo notebook dimostra come creare un **sistema di raccomandazione** utilizzando IBM Logical Neural Networks per combinare regole logiche esplicite con inferenza automatica.

## Cosa Imparerai

- Definire regole di raccomandazione basate su generi e similarità
- Inferire preferenze utente da comportamenti passati
- Usare predicati binari (arity=2) per relazioni
- Generare raccomandazioni con spiegazioni interpretabili

## Approccio

Il sistema combina due strategie:
1. **Content-based**: Raccomanda film con generi che piacciono all'utente
2. **Collaborative**: Raccomanda film simili a quelli già visti

Le regole logiche LNN integrano naturalmente entrambi gli approcci.

---

**Nota sull'architettura**: Questo notebook segue le best practice DRY. La logica del sistema è nel file `movie_recommender.py`.

## Setup Ambiente

In [ ]:
# Installazione dipendenze
!pip install -q git+https://github.com/IBM/LNN.git
!pip install -q torch>=2.0.0 numpy>=1.24.0

# Auto-download del modulo per Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("🌐 Esecuzione su Google Colab")
except ImportError:
    IN_COLAB = False
    print("💻 Esecuzione locale")

if IN_COLAB:
    import urllib.request
    url = "https://raw.githubusercontent.com/gianlucamazza/neuro_llm/main/examples/02_recommendation/movie_recommender.py"
    print("📥 Download modulo movie_recommender.py...")
    urllib.request.urlretrieve(url, "movie_recommender.py")
    print("✓ Modulo scaricato correttamente")
else:
    print("✓ Usando file locale movie_recommender.py")

print("\n" + "="*50)
print("Setup completato!")
print("="*50)

## Import Moduli

In [ ]:
from movie_recommender import MovieRecommendationSystem
from lnn import Fact

print("✓ MovieRecommendationSystem caricato correttamente")
print("✓ Pronto per creare raccomandazioni!")

## Creazione del Sistema

In [ ]:
print("="*70)
print("SISTEMA DI RACCOMANDAZIONE FILM con LNN")
print("="*70)

system = MovieRecommendationSystem()
print("\n✓ Sistema creato con successo!")

## Popolamento Catalogo

Aggiungiamo film di diversi generi.

In [ ]:
print("\n[1] Popolamento catalogo film...")

# Film SciFi
system.add_movie('Inception', ['SciFi', 'Thriller', 'Action'])
system.add_movie('Interstellar', ['SciFi', 'Drama'])
system.add_movie('The Matrix', ['SciFi', 'Action'])
system.add_movie('Tenet', ['SciFi', 'Thriller'])
system.add_movie('Arrival', ['SciFi', 'Drama'])

# Film Romance
system.add_movie('Titanic', ['Romance', 'Drama'])
system.add_movie('The Notebook', ['Romance', 'Drama'])
system.add_movie('La La Land', ['Romance', 'Musical'])
system.add_movie('Pride and Prejudice', ['Romance', 'Drama'])

# Film Crime
system.add_movie('The Godfather', ['Crime', 'Drama'])
system.add_movie('Goodfellas', ['Crime', 'Drama'])
system.add_movie('The Dark Knight', ['Action', 'Crime'])

print(f"✓ Catalogo popolato con {len(system.movies)} film")

## Definizione Similarità

Specifichiamo quali film sono simili tra loro.

In [ ]:
print("\n[2] Definizione similarità tra film...")

# SciFi similarities
system.add_movie_similarity('Inception', 'Tenet', 0.9)
system.add_movie_similarity('Inception', 'Interstellar', 0.7)
system.add_movie_similarity('Interstellar', 'Arrival', 0.8)
system.add_movie_similarity('The Matrix', 'Inception', 0.75)

# Romance similarities
system.add_movie_similarity('Titanic', 'The Notebook', 0.85)
system.add_movie_similarity('The Notebook', 'Pride and Prejudice', 0.7)

# Crime similarities
system.add_movie_similarity('The Godfather', 'Goodfellas', 0.9)

print("✓ Similarità definite")

## Aggiunta Utenti

Creiamo tre profili utente con preferenze diverse.

In [ ]:
print("\n[3] Aggiunta storico visualizzazioni...")

# Alice - Fan di SciFi
print("    Alice: Fan di SciFi")
system.add_viewing_history('Alice', ['Inception', 'Interstellar', 'The Matrix'])

# Bob - Fan di Romance
print("    Bob: Fan di Romance")
system.add_viewing_history('Bob', ['Titanic', 'The Notebook'])

# Charlie - Preferenza Crime
print("    Charlie: Preferenza esplicita per Crime")
system.add_user_preference('Charlie', 'Crime', strength=0.9)
system.add_viewing_history('Charlie', ['The Godfather'])

print("\n✓ Utenti configurati")

## Generazione Raccomandazioni

Eseguiamo inferenza e generiamo raccomandazioni personalizzate.

In [ ]:
print("\n[4] Generazione raccomandazioni...\n")

for user in ['Alice', 'Bob', 'Charlie']:
    print(f"\n{'='*70}")
    print(f"RACCOMANDAZIONI PER: {user.upper()}")
    print(f"{'='*70}")

    recommendations = system.get_recommendations(user, top_n=5)

    if recommendations:
        for i, (movie, score) in enumerate(recommendations, 1):
            explanation = system.explain_recommendation(user, movie)
            print(f"{i}. {movie:25s} [Score: {score:.2f}]")
            print(f"   -> {explanation}")
    else:
        print("Nessuna raccomandazione disponibile")

## Conclusioni

### Caratteristiche dimostrate:

1. **Inferenza automatica di preferenze** da comportamento storico
2. **Regole logiche esplicite** combinano content-based e collaborative
3. **Spiegazioni interpretabili** per ogni raccomandazione
4. **Gestione naturale incertezza** tramite bounds

### Vantaggi vs approcci tradizionali:

- **Vs Matrix Factorization**: Completamente interpretabile
- **Vs Rule-based puri**: Gestisce incertezza e può apprendere
- **Vs Deep Learning**: Richiede meno dati, più trasparente

### Possibili estensioni:

- Training per apprendere pesi regole da dati
- Integrazione con LLM per estrarre similarità da recensioni
- Regole più complesse (temporal patterns)
- Cold start handling con regole default